# 03 - Modelagem e Avaliação
## Tech Challenge Fase 1 - Saúde e Segurança da Mulher
**Responsável:** Rodrigo

---

### Objetivo
Treinar, avaliar e comparar modelos de Machine Learning para classificação de tumores mamários. Implementar explicabilidade com SHAP e discutir a aplicabilidade prática.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.data_loader import load_breast_cancer_dataset
from src.preprocessing import full_preprocessing_pipeline
from src.models import create_models, train_all_models, save_model
from src.evaluation import (
    evaluate_all_models,
    plot_confusion_matrix,
    plot_roc_curves,
    explain_with_shap,
    get_feature_importance
)

%matplotlib inline

## 1. Preparação dos Dados

In [ ]:
# Carregar e preprocessar
df = load_breast_cancer_dataset()

pipeline = full_preprocessing_pipeline(
    df, target_col='target', drop_cols=['diagnosis'], test_size=0.2
)

X_train = pipeline['X_train']
X_test = pipeline['X_test']
y_train = pipeline['y_train']
y_test = pipeline['y_test']
feature_names = pipeline['feature_names']

## 2. Criação e Treinamento dos Modelos

In [ ]:
# Criar modelos
models = create_models()

# Treinar todos
trained_models = train_all_models(models, X_train, y_train)

## 3. Avaliação dos Modelos

In [ ]:
# Avaliar todos os modelos
results_df = evaluate_all_models(trained_models, X_test, y_test)
results_df

In [ ]:
# Salvar resultados
results_df.to_csv('../results/comparacao_modelos.csv')
print("Resultados salvos em results/comparacao_modelos.csv")

## 4. Matrizes de Confusão

In [ ]:
# Matriz de confusão para cada modelo
for name, model in trained_models.items():
    save_path = f'../reports/figures/confusion_matrix_{name.replace(" ", "_")}.png'
    plot_confusion_matrix(model, X_test, y_test, name, save_path=save_path)

## 5. Curvas ROC

In [ ]:
# Curvas ROC comparativas
plot_roc_curves(trained_models, X_test, y_test,
                save_path='../reports/figures/curvas_roc.png')

## 6. Feature Importance

In [ ]:
# Feature importance para Random Forest
get_feature_importance(
    trained_models['Random Forest'], feature_names,
    'Random Forest',
    save_path='../reports/figures/feature_importance_rf.png'
)

## 7. Explicabilidade com SHAP

In [ ]:
# SHAP para Random Forest
shap_values = explain_with_shap(
    trained_models['Random Forest'], X_test, feature_names,
    model_name='Random Forest',
    save_dir='../reports/figures'
)

In [ ]:
# SHAP para Regressão Logística
shap_values_lr = explain_with_shap(
    trained_models['Regressão Logística'], X_test, feature_names,
    model_name='Regressão Logística',
    save_dir='../reports/figures'
)

## 8. Salvar Melhor Modelo

In [ ]:
# Identificar e salvar o melhor modelo (por recall)
best_model_name = results_df['recall'].idxmax()
print(f"Melhor modelo (por Recall): {best_model_name}")
print(f"Recall: {results_df.loc[best_model_name, 'recall']:.4f}")

save_model(trained_models[best_model_name], '../results/best_model.joblib')

## 9. Discussão Crítica dos Resultados

*(Rodrigo: preencher com a análise crítica)*

### 9.1 Comparação dos Modelos
- Qual modelo teve melhor desempenho e por quê?
- Por que priorizamos o Recall em diagnóstico médico?

### 9.2 Interpretação dos Resultados (SHAP)
- Quais features mais influenciam o diagnóstico?
- Os resultados fazem sentido clinicamente?

### 9.3 Aplicabilidade Prática
- O modelo pode ser utilizado na prática clínica? Como?
- **IMPORTANTE:** O modelo é uma ferramenta de APOIO. O médico sempre deve ter a palavra final.
- Quais são as limitações do modelo?
- Como o sistema se integraria no fluxo de trabalho hospitalar?

### 9.4 Limitações
- Tamanho do dataset
- Possíveis vieses
- Generalização para outras populações